# Online Retail Product Revenue Analysis: Q1 2010

### Notebook Overview
This notebook processes the "Online Retail" dataset from the UCI Machine Learning Repository. The primary goal is to isolate transactions from the first quarter of 2010, calculate the revenue for each product sold during that period, and create a final summary table showing the total revenue per product.

## Table of Contents

1.  [**Setup and Initialization**](#setup)
    *   *This section handles the initial setup of the analysis environment. It involves importing essential Python libraries such as `pandas` for data manipulation and `numpy` for numerical operations.*

2.  [**Data Loading**](#data-loading)
    *   *Here, the raw "Online Retail" dataset is loaded from its source file (e.g., an Excel or CSV file) into a pandas DataFrame, making it accessible for processing.*

3.  [**Data Cleaning and Preprocessing**](#data-preprocessing)
    *   *This critical step addresses data quality issues. It includes tasks like handling missing values (e.g., in `Description` or `CustomerID`), ensuring data types are correct (e.g., converting `InvoiceDate` from an object to a datetime format), and removing any invalid or cancelled transactions.*

4.  [**Temporal Filtering for Q1 2010**](#temporal-filtering)
    *   *To focus the analysis, the DataFrame is filtered to create a temporal subset. This section isolates only the transactions that occurred within the first quarter of the year 2010 (from January 1st to March 31st).*

5.  [**Feature Engineering: Revenue Calculation**](#feature-engineering)
    *   *A new feature, `item_total`, is engineered to facilitate the analysis. This is calculated for each transaction line by multiplying the `Quantity` of items sold by their `UnitPrice`.*

6.  [**Data Aggregation by Product**](#data-aggregation)
    *   *The core transformation of the analysis happens here. The data is grouped by product (`StockCode` and `Description`) and the `Revenue` is summed up for each group to compute the total revenue generated by each unique product in Q1 2010.*

7.  [**Final Data Representation and Export**](#final-data)
    *   *This final section displays the resulting aggregated DataFrame, which represents the clean, final dataset showing the total revenue per product. This is the data representation you mentioned, ready for visualization, further analysis, or modeling.*

## Read cleaned (denoised) data, post EDA

In [1]:
from importlib.resources import files
import pandas as pd
fp = str(files("kmds.examples").joinpath("retail_q1_post_EDA.parquet"))
df = pd.read_parquet(fp)

## Develop daily sales data for store inventory
The daily sales data for the store inventory for Q1 was generated by computing sales for each inventory item for each business day of Q1

In [2]:
df["item_total"] = df["Quantity"] * df["Price"]
dsbysc = df.groupby([df.InvoiceDate.dt.day_of_year, df.StockCode])
dsbysc = dsbysc["item_total"].sum().to_frame().reset_index()
dfQ1_PA = dsbysc.pivot(index="InvoiceDate", columns="StockCode", values="item_total").fillna(0)

## Write the transformed data representation for use in modelling

In [3]:
fp = str(files("kmds.examples").joinpath("retail_q1_post_data_rep_prep.parquet"))
dfQ1_PA.to_parquet(fp, index=False)

## Log the data transformation information to KMDS

In [4]:
from kmds.tagging.tag_types import DataRepresentationTags
from owlready2 import *
from kmds.utils.load_utils import *
from kmds.utils.path_utils import *
KNOWLEDGE_BASE = str(files("kmds.examples").joinpath("example_ml_kb_exp_workflow.xml"))


* Owlready2 * Warning: optimized Cython parser module 'owlready2_optimized' is not available, defaulting to slower Python implementation


In [5]:
onto2 = load_kb(KNOWLEDGE_BASE)


In [6]:
with onto2:
    insts = Workflow.instances()

In [7]:
the_workflow_instance = insts[0]

In [8]:
from kmds.ontology.intent_types import IntentType
dr_obs_list = []
observation_count = 1

dr1 = DataRepresentationObservation(namespace=onto2)
dr1.finding = "This data is from https://archive.ics.uci.edu/dataset/352/online+retail.\
The subset of transactions that fall in the first quarter of 2010 is filtered.\
Then the revenue for each product is computed. This is the data represenation for this example."
dr1.finding_sequence = observation_count
dr1.data_representation_observation_type = DataRepresentationTags.DATA_TRANSFORMATION_OBSERVATION.value
dr1.intent = IntentType.DATA_UNDERSTANDING.value
dr_obs_list.append(dr1)
the_workflow_instance.has_data_representation_observations = dr_obs_list

onto2.save(file=KNOWLEDGE_BASE, format="rdfxml")